In [ ]:
# Geology 593 - Engineering Geology
# Module 4 - Cleaning and Visualizing Site Investigation Data

In [ ]:
#Import Python Libraries
!pip install pandas openpyxl xlrd

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Import XLS files
from google.colab import files
uploaded = files.upload()

Saving 24-61-27761_CP24-COOK-01-BSC.XLS to 24-61-27761_CP24-COOK-01-BSC (2).XLS
Saving 24-61-27761_CP24-COOK-03-BSC.XLS to 24-61-27761_CP24-COOK-03-BSC (2).XLS
Saving 24-61-27761_CP24-COOK-04-BSC.XLS to 24-61-27761_CP24-COOK-04-BSC (2).XLS
Saving 24-61-27761_CP24-COOK-06-BSC.XLS to 24-61-27761_CP24-COOK-06-BSC (2).XLS
Saving 24-61-27761_CP24-COOK-06-OFF01-BSC.XLS to 24-61-27761_CP24-COOK-06-OFF01-BSC (2).XLS
Saving 24-61-27761_SP24-COOK-02-BSC.XLS to 24-61-27761_SP24-COOK-02-BSC (2).XLS
Saving 24-61-27761_SP24-COOK-02-OFF01-BSC.XLS to 24-61-27761_SP24-COOK-02-OFF01-BSC (2).XLS
Saving 24-61-27761_SP24-COOK-02-OFF02-BSC.XLS to 24-61-27761_SP24-COOK-02-OFF02-BSC (2).XLS
Saving 24-61-27761_SP24-COOK-07-BSC.XLS to 24-61-27761_SP24-COOK-07-BSC (2).XLS


In [ ]:
# Inpect the workbook
path = "24-61-27761_CP24-COOK-01-BSC.XLS"

# See available sheet names (if multiple)
xl = pd.ExcelFile(path, engine="xlrd")
print(xl.sheet_names)

# Peek at the top of the sheet (often metadata)
preview = pd.read_excel(path, sheet_name=0, nrows=30, header=None, engine="xlrd")
print(preview)


['Sheet1']
     0                                                  1   2   3  \
0  NaN                  ConeTec Calculated CPT Parameters NaN NaN   
1  NaN  Calculated CPT Parameters Output - SCREENzW Ve... NaN NaN   
2  NaN                                 DAS Version: CS162 NaN NaN   
3  NaN                             Interpretation Format: NaN NaN   
4  NaN                                            Run ID: NaN NaN   
5  NaN                                            Job No: NaN NaN   
6  NaN                                            Client: NaN NaN   
7  NaN                                           Project: NaN NaN   
8  NaN                                          Facility: NaN NaN   
9  NaN                                       Sounding ID: NaN NaN   
10 NaN                                           Cone ID: NaN NaN   
11 NaN                                          Operator: NaN NaN   
12 NaN                                          CPT Date: NaN NaN   
13 NaN                 

In [ ]:
# Getting the data table
def find_header_row(path, sheet_name=0, search_text="Depth", max_rows=120):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=max_rows)
    mask = raw.apply(
        lambda row: row.astype(str).str.contains(search_text, case=False, na=False).any(),
        axis=1
    )
    hits = raw.index[mask].tolist()
    return hits[0] if hits else None

header_row = find_header_row(path, sheet_name=0, search_text="Depth")
print("Detected header row:", header_row)

df = pd.read_excel(path, sheet_name=0, skiprows=header_row, header=0)
print(df.columns)
print(df.head())


Detected header row: 38
Index(['Layer', 'Depth', 'Depth.1', 'qc', 'qt', 'fs', 'u', 'Rf'], dtype='object')
   Layer  Depth  Depth.1      qc         qt     fs      u          Rf
0    NaN      m       ft     tsf        tsf    tsf     ft           %
1    1.0  0.025  0.08202   0.036    0.03673  0.038  0.117  103.456543
2    2.0   0.05  0.16404  21.657  21.678931   0.08  3.513    0.369022
3    3.0  0.075  0.24606  41.663  41.696486  0.097  5.364    0.232634
4    4.0    0.1  0.32808  64.817  64.861517  0.201  7.131    0.309891


In [ ]:
# Cleaning and Standardizing CPT Data in pandas

# Clean columns names
df.columns = [str(c).strip() for c in df.columns]

# Rename columns to standard names
rename_map = {
    "Depth": "depth_m",
    "Depth.1": "depth_ft",
    "qc": "qc_tsf",
    "qt": "qt_tsf",
    "fs": "fs_tsf",
    "u": "u_tsf",
    "Rf": "rf_pct"
}
df = df.rename(columns=rename_map)
print(df.rename)

# Convert numeric columns into real numbers
num_cols = ["depth_ft", "depth_m", "qc_tsf", "qt_tsf", "fs_tsf", "u_tsf", "rf_pct"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows with no detph
if "depth_ft" in df.columns:
    df = df.dropna(subset=["depth_ft"])

# Drop empty rows
df = df.dropna(how="all")

# Add sounding ID
sounding_id = "CPT24-cook-01"
df["sounding_id"] = sounding_id

# Sort by depth
print(df.columns)
depth_col = "depth_ft" if "depth_ft" in df.columns else "depth_m"
df = df.sort_values(depth_col).reset_index(drop=True)

# Sort by depth to ensure consistent plotting/analysis
depth_col = "depth_ft" if "depth_ft" in df.columns else "depth_m"
df = df.sort_values(depth_col).reset_index(drop=True)

# Quick checks
print("Min depth:", df[depth_col].min(), "Max depth:", df[depth_col].max())
print("Has duplicates?", df[depth_col].duplicated().any())

# Example rule: negative sleeve friction is suspicious; flag it
if "fs_tsf" in df.columns:
    df["fs_negative_flag"] = df["fs_tsf"] < 0

# Example rule: friction ratio should not be negative; set negatives to NaN
if "rf_pct" in df.columns:
    df.loc[df["rf_pct"] < 0, "rf_pct"] = pd.NA


# Only compute if you have both columns and qc is nonzero
if ("fs_tsf" in df.columns) and ("qc_tsf" in df.columns):
    df["rf_pct_calc"] = (df["fs_tsf"] / df["qc_tsf"]) * 100
    df.loc[df["qc_tsf"] == 0, "rf_pct_calc"] = pd.NA





<bound method DataFrame.rename of      Layer  depth_m  depth_ft   qc_tsf      qt_tsf  fs_tsf   u_tsf  \
0      1.0    0.025   0.08202    0.036    0.036730   0.038   0.117   
1      2.0    0.050   0.16404   21.657   21.678931   0.080   3.513   
2      3.0    0.075   0.24606   41.663   41.696486   0.097   5.364   
3      4.0    0.100   0.32808   64.817   64.861517   0.201   7.131   
4      5.0    0.125   0.41010   82.667   82.724046   0.490   9.138   
..     ...      ...       ...      ...         ...     ...     ...   
448  449.0   11.225  36.82698   85.012   85.174604   2.374  26.047   
449  450.0   11.250  36.90900  107.297  107.494638   0.000  31.659   
450  451.0   11.275  36.99102  169.366  169.486079   0.000  19.235   
451  452.0   11.300  37.07304  224.920  225.134487   0.000  34.358   
452  453.0   11.325  37.15506  320.828  321.021287   0.000  30.962   

         rf_pct    sounding_id  
0    103.456543  CPT24-cook-01  
1      0.369022  CPT24-cook-01  
2      0.232634  CPT24-coo